# The Median Fan — 10/5/2-year median lines as a drawdown-depth ruler
**Jake's idea.** Fit a **median (LAD) regression** through log-price at three lookbacks — **10y, 5y, 2y** — to
get a *fan* of trend lines. The **spread between them = the trend's acceleration** (2y steeper than 10y = growth
sped up = price extended above its long-run path). Then measure **how deep past drawdowns fell below each line**,
build a *distribution* of where pullbacks bottom, and read **where we are now** against it.

### What it answers
1. Where is price **right now** vs its 10/5/2y median lines (in % and robust σ)?
2. Where did **each past drawdown** bottom relative to each line (2022, 2023, 2024, last year…)?
3. Given that history, if *this* pullback bottoms where a **typical / deep** one did, **how far down is that**?
4. How fast are the lines growing/spreading (the acceleration signal)?

### ⚠️ Read before trusting
- **Tiny sample.** ~6–12 real drawdowns in 15y, and they are **NOT independent** (regimes cluster). "Typical
  bottom depth" is a *distribution*, not a target. Treat outputs as odds-shaders, never a called bottom.
- **Descriptive, not a timer** (WARNING-vs-TRIGGER rule): deviation-below-a-line is a *state*. It shifts the
  ODDS of how much further a pullback runs; it does not *time* the bottom. A cheap market can get cheaper.
- **Index-level = no survivorship bias** (unlike the per-name study) — SPY *is* the index. Clean here.
- The median line beat SMA/rolling-median as a **short-horizon dip trigger** in the per-name study
  ([[median-line-dip]]); this notebook asks a different question — *drawdown depth*, not entry timing.


In [ ]:
# 1 · Setup
!pip -q install yfinance >/dev/null 2>&1
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
import yfinance as yf, matplotlib.pyplot as plt
try:
    import statsmodels.api as sm; HAVE_SM=True
except Exception: HAVE_SM=False
try:
    from scipy.stats import theilslopes; from scipy.optimize import linprog; HAVE_SCIPY=True
except Exception: HAVE_SCIPY=False
print("statsmodels",HAVE_SM,"| scipy",HAVE_SCIPY)


In [ ]:
# 2 · CONFIG
TICKER      = "SPY"        # index proxy. Any ticker works; SPY/^GSPC for 'where are we'
YEARS_PULL  = 15           # history to download (need >10y so the 10y line has runway)
HORIZONS    = [10, 5, 2]   # median-line lookbacks, in years (the fan)
LINE_METHOD = "lad"        # "lad" (median regression) | "theil" | "ols"
REFIT_DAYS  = 21           # refit each line every N trading days
RESAMPLE    = "W"          # weekly resample for the line fit (speed)
DD_THRESH   = 0.05         # a 'drawdown episode' = peak-to-trough decline ≥ this (5%)
DEEP_PCTL   = 20           # 'deep' bottom = this percentile of historical trough depth (20th = deeper)
print("CONFIG:", TICKER, HORIZONS, LINE_METHOD)


In [ ]:
# 3 · Median line at an arbitrary trailing window
def fit_line(x, y):
    x=np.asarray(x,float); y=np.asarray(y,float)
    if len(x)<5: return np.nan,np.nan
    if LINE_METHOD=="lad" and HAVE_SM:
        try:
            r=sm.QuantReg(y, sm.add_constant(x)).fit(q=0.5); return float(r.params[0]),float(r.params[1])
        except Exception: pass
    if LINE_METHOD=="lad" and HAVE_SCIPY:
        try:
            n=len(x); c=np.r_[0,0,np.ones(n)]; A=np.zeros((2*n,2+n)); bnd=np.zeros(2*n)
            for i in range(n):
                A[i,0]=-1;A[i,1]=-x[i];A[i,2+i]=-1;bnd[i]=-y[i]
                A[n+i,0]=1;A[n+i,1]=x[i];A[n+i,2+i]=-1;bnd[n+i]=y[i]
            r=linprog(c,A_ub=A,b_ub=bnd,bounds=[(None,None)]*2+[(0,None)]*n,method="highs")
            if r.success: return float(r.x[0]),float(r.x[1])
        except Exception: pass
    if LINE_METHOD in ("lad","theil") and HAVE_SCIPY:
        try:
            b,a,*_=theilslopes(y,x); return float(a),float(b)
        except Exception: pass
    b,a=np.polyfit(x,y,1); return float(a),float(b)

def rscale(r):
    r=r[np.isfinite(r)]
    return (1.4826*np.median(np.abs(r-np.median(r))) or np.nan) if len(r)>=5 else np.nan

def median_line(close, win_years):
    '''Walk-forward line/scale/z/slope for a trailing win_years window. No lookahead.'''
    lp=np.log(close.astype(float)); idx=lp.index
    line=pd.Series(np.nan,index=idx); scale=pd.Series(np.nan,index=idx); slope=pd.Series(np.nan,index=idx)
    yrs=pd.Series((idx-idx[0]).days/365.25, index=idx)
    start=np.searchsorted(idx, idx[0]+pd.Timedelta(days=int(win_years*365.25)))
    pts=list(range(start,len(idx),REFIT_DAYS))
    if pts and pts[-1]!=len(idx)-1: pts.append(len(idx)-1)
    for k,ri in enumerate(pts):
        d0=idx[ri]-pd.Timedelta(days=int(win_years*365.25))
        win=lp[(idx>=d0)&(idx<=idx[ri])]
        if len(win)<40: continue
        w=win.resample(RESAMPLE).last().dropna() if RESAMPLE else win
        wx=(w.index-idx[0]).days/365.25
        a,b=fit_line(wx,w.values)
        if not np.isfinite(a): continue
        s=rscale(w.values-(a+b*wx))
        lo=ri; hi=pts[k+1] if k+1<len(pts) else len(idx); seg=idx[lo:hi]
        line.loc[seg]=a+b*yrs.loc[seg].values; scale.loc[seg]=s; slope.loc[seg]=b
    z=(lp-line)/scale
    return dict(line_log=line, scale=scale, z=z, slope=slope)  # slope = log-growth per year


In [ ]:
# 4 · Data + the three lines
px = yf.download(TICKER, period=f"{YEARS_PULL}y", auto_adjust=True, progress=False)["Close"]
px = px.dropna()
if hasattr(px,'columns'): px = px.iloc[:,0]
print(f"{TICKER}: {len(px)} days, {px.index[0].date()} → {px.index[-1].date()}  last={px.iloc[-1]:.2f}")
FAN = {h: median_line(px, h) for h in HORIZONS}


In [ ]:
# 5 · Drawdown episodes (peak→trough ≥ DD_THRESH) + depth below each line at the trough
# Episode = a contiguous 'underwater' stretch (price below its running peak); kept if the deepest
# point is ≤ -DD_THRESH. peak = last all-time-high before the trough; trough = the min inside it.
peak = px.cummax()
dd = px/peak - 1.0
episodes=[]; i=0; n=len(px); arr=dd.values; idx=px.index
while i<n:
    if arr[i]<0:
        j=i
        while j<n and arr[j]<0: j+=1          # extend to the end of this underwater stretch
        seg=dd.iloc[i:j]
        if seg.min()<=-DD_THRESH:
            tdate=seg.idxmin()
            pk_date=px[:tdate][px[:tdate]==px[:tdate].cummax()].index[-1]
            episodes.append(dict(peak=pk_date, trough=tdate, end=idx[min(j,n-1)],
                                 depth=float(seg.min())))
        i=j
    else: i+=1

rows=[]
for e in episodes:
    t=e["trough"]; r={"peak":e["peak"].date(),"trough":t.date(),"drawdown":f"{e['depth']:.1%}"}
    for h in HORIZONS:
        z=FAN[h]["z"].get(t, np.nan); r[f"z_{h}y"]=round(z,2) if np.isfinite(z) else np.nan
    rows.append(r)
DD=pd.DataFrame(rows)
print(f"\n{len([e for e in episodes if e['depth']<=-DD_THRESH])} drawdowns ≥{DD_THRESH:.0%} since {px.index[0].date()}")
print("  trough deviation (σ) below each median line:\n")
print(DD.to_string(index=False) if len(DD) else "none")


In [ ]:
# 6 · Distribution of where drawdowns bottom (σ below each line)  + the FAN / acceleration now
print("="*68); print("WHERE DRAWDOWNS BOTTOM — trough z distribution per median line"); print("="*68)
for h in HORIZONS:
    zc=DD[f"z_{h}y"].dropna()
    if len(zc)<2: print(f" {h}y: thin"); continue
    print(f" {h}y line:  median trough {zc.median():+.2f}σ | deep({DEEP_PCTL}th pctl) "
          f"{np.percentile(zc,DEEP_PCTL):+.2f}σ | worst {zc.min():+.2f}σ  (n={len(zc)})")

print("\n"+"="*68); print("THE FAN NOW — growth rate + where price sits vs each line"); print("="*68)
last=px.iloc[-1]; lastd=px.index[-1]
for h in HORIZONS:
    f=FAN[h]; sl=f["slope"].dropna().iloc[-1]; ln=np.exp(f["line_log"].dropna().iloc[-1])
    z=f["z"].dropna().iloc[-1]
    print(f" {h}y line: growth {np.exp(sl)-1:+.1%}/yr | line≈{ln:.2f} | price {last:.2f} "
          f"= {z:+.2f}σ ({(last/ln-1):+.1%} vs line)")
s2=FAN[2]['slope'].dropna().iloc[-1]; s10=FAN[10]['slope'].dropna().iloc[-1]
acc=np.exp(s2)-np.exp(s10)
print(f"\n ACCELERATION (2y growth − 10y growth): {acc:+.1%}/yr  "
      f"→ {'ACCELERATING (extended above long-run path)' if acc>0 else 'DECELERATING'}")


In [ ]:
# 7 · WHERE WE ARE NOW → implied downside IF this pullback bottoms like history did
print("="*68); print(f"{TICKER} = {last:.2f} on {lastd.date()}  —  implied bottoms by median-fan history")
print("(DESCRIPTIVE: where past pullbacks stopped vs each line, applied to today's lines. NOT a target.)")
print("="*68)
for h in HORIZONS:
    f=FAN[h]; zc=DD[f"z_{h}y"].dropna()
    ln=f["line_log"].dropna().iloc[-1]; sc=f["scale"].dropna().iloc[-1]; znow=f["z"].dropna().iloc[-1]
    if len(zc)<2: print(f" {h}y: thin history"); continue
    for label,zt in [("typical", zc.median()), (f"deep({DEEP_PCTL}th)", np.percentile(zc,DEEP_PCTL))]:
        price_at=np.exp(ln+zt*sc); dn=price_at/last-1
        note="  (already below this)" if znow<=zt else ""
        print(f" {h}y {label:<11} bottom ≈ {zt:+.2f}σ → {TICKER} ~{price_at:.0f}  ({dn:+.1%} from here){note}")
    print(f"   now: {znow:+.2f}σ vs the {h}y line\n")
print("Read: 'typical' = median past trough vs that line; 'deep' = the 20th-pctl (worse) trough.")
print("If price already sits below a line's typical-bottom σ, that horizon says the easy downside is done.")


In [ ]:
# 8 · SEE the fan: price + 3 median lines + drawdown troughs + now
fig,ax=plt.subplots(2,1,figsize=(13,8),sharex=True,gridspec_kw={"height_ratios":[3,1]})
ax[0].plot(px.index,px.values,color="black",lw=1,label=TICKER)
colors={10:"tab:blue",5:"tab:green",2:"tab:red"}
for h in HORIZONS:
    ln=np.exp(FAN[h]["line_log"]); ax[0].plot(ln.index,ln.values,lw=1.5,color=colors[h],alpha=.85,label=f"{h}y median line")
for e in episodes:
    if e["depth"]<=-DD_THRESH:
        ax[0].scatter([e["trough"]],[px.loc[e["trough"]]],color="red",s=28,zorder=5)
ax[0].scatter([lastd],[last],color="gold",edgecolor="k",s=90,zorder=6,label="now")
ax[0].set_yscale("log"); ax[0].set_title(f"{TICKER} — the median fan (10/5/2y) + drawdown troughs"); ax[0].legend(fontsize=8)
for h in HORIZONS:
    ax[1].plot(FAN[h]["z"].index,FAN[h]["z"].values,lw=.8,color=colors[h],label=f"z vs {h}y")
ax[1].axhline(0,color="k",lw=.5); ax[1].set_ylabel("σ vs line"); ax[1].legend(fontsize=7,ncol=3)
plt.tight_layout(); plt.show()


In [ ]:
# 8b · SPREAD DYNAMICS — the growth rate of the GAP between the 10/5/2y lines during drawdowns
# Jake's metric: NOT the slope of any one line — how fast the SPREAD between the lines changes in a drawdown.
ROC_WIN = 21   # window for 'growth rate of the spread' (≈1 month, trading days)
Lg = {h: FAN[h]["line_log"] for h in HORIZONS}
# signed spreads in %: LONGER line minus SHORTER line. Positive = short line BELOW long line = drawdown-like fan.
spread = pd.DataFrame(index=px.index)
spread["s10_2"] = (np.exp(Lg[10])/np.exp(Lg[2]) - 1)*100   # 10y above 2y (post-2021 only)
spread["s10_5"] = (np.exp(Lg[10])/np.exp(Lg[5]) - 1)*100
spread["s5_2"]  = (np.exp(Lg[5]) /np.exp(Lg[2]) - 1)*100   # most history (post-2016)
vel = spread.diff(ROC_WIN)                                  # growth rate of the spread, %-pts / month
PRIMARY = "s5_2"   # the pair with the most drawdown coverage; s10_2 shown where it exists
idxpos = {d:i for i,d in enumerate(px.index)}

rows=[]
for e in episodes:
    if e["depth"]>-DD_THRESH: continue
    pk,tr=e["peak"],e["trough"]; ip=idxpos[pk]
    s_pk=spread[PRIMARY].get(pk,np.nan); s_tr=spread[PRIMARY].get(tr,np.nan)
    ev = vel[PRIMARY].iloc[ip+ROC_WIN] if ip+ROC_WIN < len(px) else np.nan  # spread growth in 1st month of decline
    vseg = vel[PRIMARY].loc[pk:tr].dropna()
    vmax = vseg.max() if len(vseg) else np.nan
    vmax_date = vseg.idxmax() if len(vseg) else None
    lead = (tr - vmax_date).days if vmax_date is not None else np.nan   # +ve = spread-velocity peaked BEFORE trough
    rows.append(dict(trough=tr.date(), depth=f"{e['depth']:.1%}",
                     s52_peak=round(s_pk,2) if np.isfinite(s_pk) else np.nan,
                     s52_trough=round(s_tr,2) if np.isfinite(s_tr) else np.nan,
                     d_spread=round(s_tr-s_pk,2) if np.isfinite(s_tr) and np.isfinite(s_pk) else np.nan,
                     early_vel=round(ev,2) if np.isfinite(ev) else np.nan,
                     peakvel=round(vmax,2) if np.isfinite(vmax) else np.nan,
                     vel_lead_d=int(lead) if np.isfinite(lead) else np.nan))
SD=pd.DataFrame(rows)
print("="*72); print(f"SPREAD DYNAMICS per drawdown  (pair={PRIMARY}, velocity over {ROC_WIN}d, %-pts/mo)")
print("  d_spread = fan opening peak→trough | early_vel = spread growth in 1st month | vel_lead_d = days velocity peaked BEFORE the trough")
print("="*72)
print(SD.to_string(index=False) if len(SD) else "none")


In [ ]:
# 8c · Does the spread's growth rate say anything? (correlations + lead) — small-n, read as suggestive
def spearman(a,b):
    a=pd.Series(a); b=pd.Series(b); m=a.notna()&b.notna()
    if m.sum()<4: return np.nan,0
    ar=a[m].rank(); br=b[m].rank()
    return float(np.corrcoef(ar,br)[0,1]), int(m.sum())
depths=[float(e["depth"]) for e in episodes if e["depth"]<=-DD_THRESH]
ev=SD["early_vel"].tolist(); pv=SD["peakvel"].tolist(); lead=SD["vel_lead_d"].dropna()
r1,n1=spearman(ev,depths); r2,n2=spearman(pv,depths)
print("="*72); print("DOES SPREAD-GROWTH-RATE RELATE TO DRAWDOWN DEPTH? (Spearman; -1 = faster fan → deeper)")
print("="*72)
print(f" early-month spread velocity  vs depth:  rho={r1:+.2f}  (n={n1})" if np.isfinite(r1) else " early-vel: thin")
print(f" peak spread velocity         vs depth:  rho={r2:+.2f}  (n={n2})" if np.isfinite(r2) else " peakvel: thin")
if len(lead): print(f"\n spread-velocity peaked a median of {lead.median():.0f} days BEFORE the trough "
                    f"(range {int(lead.min())}–{int(lead.max())}d) → potential bottom lead if positive & consistent")
print("\n ⚠️ n is tiny and episodes overlap regimes — a strong rho here is a hypothesis, not a validated signal.")

print("\n" + "="*72); print("THE SPREAD NOW — is the fan OPENING (drawdown building) or CLOSING (healing)?"); print("="*72)
for c,lab in [("s5_2","5y−2y"),("s10_2","10y−2y"),("s10_5","10y−5y")]:
    s=spread[c].dropna(); v=vel[c].dropna()
    if not len(s): print(f" {lab}: thin"); continue
    arrow = "OPENING ↑ (2y rolling under — drawdown-building)" if v.iloc[-1]>0 else "closing ↓ (2y leading up — healing)"
    print(f" {lab}: spread {s.iloc[-1]:+.2f}% | Δ/mo {v.iloc[-1]:+.2f}pts → {arrow}")
evd=SD['early_vel'].dropna()
if len(evd):
    now=vel[PRIMARY].dropna().iloc[-1]
    print(f"\n current {PRIMARY} velocity {now:+.2f} vs drawdown-onset early-vel median {evd.median():+.2f} "
          f"→ {'IN the drawdown-onset range' if now>=evd.median() else 'below onset range (no fan-opening stress yet)'}")


In [ ]:
# 8d · SEE the spread + its velocity, drawdowns shaded
fig,ax=plt.subplots(2,1,figsize=(13,7),sharex=True)
for c,lab,col in [("s5_2","5y−2y","tab:purple"),("s10_2","10y−2y","tab:blue"),("s10_5","10y−5y","tab:green")]:
    ax[0].plot(spread.index,spread[c],lw=1,color=col,label=lab)
ax[0].axhline(0,color="k",lw=.5); ax[0].set_ylabel("spread %  (long−short)"); ax[0].legend(fontsize=8)
ax[0].set_title("Fan spread (positive = short line below long = drawdown-like)")
ax[1].plot(vel.index,vel[PRIMARY],lw=.8,color="tab:purple"); ax[1].axhline(0,color="k",lw=.5)
ax[1].set_ylabel(f"spread velocity\n({PRIMARY}, pts/mo)")
for e in episodes:
    if e["depth"]<=-DD_THRESH:
        ax[0].axvspan(e["peak"],e["trough"],color="red",alpha=.06)
        ax[1].axvspan(e["peak"],e["trough"],color="red",alpha=.06)
plt.tight_layout(); plt.show()


## How to read it
- **Cell 5 table** — every drawdown dated, with how many σ below each line it *bottomed*. This is the core:
  scan the `z_10y / z_5y / z_2y` columns to see the **range** where pullbacks actually stopped. 2022 vs 2023
  vs last year sit right there to compare.
- **Cell 6** — the *distribution* (median / deep / worst trough σ per line) + the **fan now**: each line's
  growth rate and where price sits. **Acceleration** = 2y growth minus 10y growth; positive = price is riding
  above its long-run path (more room to fall to the 10y line), negative = the trend is cooling.
- **Cell 7** — applies the historical trough σ to *today's* lines → an implied price if this pullback bottoms
  "typically" or "deep." **This is a distribution read, not a call.** If price already sits below a line's
  typical-bottom σ, that horizon is saying the ordinary downside is spent.
- **Knobs:** `DD_THRESH` (how big a dip counts — 0.05 vs 0.10 changes the sample), `LINE_METHOD` (lad/theil/ols),
  `DEEP_PCTL`. **Small-n honesty:** with ~6–12 episodes this is suggestive, not proven — don't over-fit the
  threshold to make a number look clean.
- Reliability question you actually asked — *is the median line a better bottom-ruler?* Compare the **spread**
  of trough σ per line: the line whose troughs cluster **tightest** (smallest range) is the most reliable
  depth-ruler. A line where bottoms scatter from −1σ to −5σ tells you little; one that consistently bottoms
  near −2σ is a usable ruler.

### The SPREAD DYNAMICS cells (8b–8d) — the growth rate of the gap between lines
- **8b** — per drawdown: the fan's spread (long−short) at peak vs trough, how much it opened (`d_spread`), the
  spread's **growth rate in the first month** (`early_vel`), its peak velocity, and `vel_lead_d` = how many days
  the spread-velocity peaked **before** the trough.
- **8c** — the payoff test: **Spearman rho of early/peak spread-velocity vs eventual depth.** Negative rho =
  *a faster-opening fan predicts a deeper drawdown* (your hypothesis). And whether the velocity **peaks before
  the bottom** (a positive, consistent `vel_lead_d` = a potential bottom lead indicator). ⚠️ n is tiny.
- **8d** — the picture: the three spreads over time + the velocity, drawdowns shaded, so you can watch the fan
  blow open on the way down and roll over near bottoms.
- **The now-line in 8c** tells you if the fan is currently **OPENING** (2y rolling under the long lines →
  drawdown-building) or **CLOSING** (healing), and whether current velocity is already in the drawdown-onset range.
- Data limit: the 10y−anything spread only exists post-2021; `s5_2` (5y−2y) carries the most episodes, so it's
  the primary pair for the stats.
